# Infant State Recognition System - Phase 1 Pipeline
**Leak-Free Evaluation: Split originals first, then augment only the training set**

This notebook runs the complete Phase 1 classical ML pipeline.

**Google Drive persistence:** All data, models, and results are saved to Drive so Phase 2 can reuse them without re-uploading.

---
## Step 0: Setup — Mount Drive & Upload Project (one-time)

In [ ]:
import os
from pathlib import Path

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Project lives on Drive so it persists across sessions
DRIVE_PROJECT = '/content/drive/MyDrive/Infant-State-Recognition-System'

if os.path.exists(os.path.join(DRIVE_PROJECT, 'src', 'config.py')):
    # Already set up from a previous session
    print(f'Project already exists on Drive: {DRIVE_PROJECT}')
    print('Skipping upload — reusing existing data, models, and code.')
    print('To re-upload, delete the folder from Drive first.')
else:
    # First time: upload and extract the zip to Drive
    from google.colab import files
    print('Upload the project zip file (infant_cry_colab.zip):')
    uploaded = files.upload()

    import zipfile
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content/drive/MyDrive/')

    print(f'Extracted to {DRIVE_PROJECT}')

# Set working directory
PROJECT_DIR = DRIVE_PROJECT
os.chdir(PROJECT_DIR)
print(f'\nWorking directory: {os.getcwd()}')
print(f'Contents: {sorted(os.listdir("."))}')

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q librosa==0.10.1 hmmlearn==0.3.0 xgboost>=2.0.0 imbalanced-learn>=0.11.0 soundfile==0.12.1
print('Dependencies installed.')

---
## Step 2: Imports & Configuration

In [ ]:
import sys
import gc
import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm import tqdm
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from src.config import (
    CLASSES, CLASS_TO_IDX, IDX_TO_CLASS, NUM_CLASSES,
    RANDOM_STATE, TEST_SIZE,
    SAMPLE_RATE, DURATION, N_SAMPLES,
    FEATURES_DIR, MODELS_DIR, RESULTS_DIR, PLOTS_DIR, METRICS_DIR,
    N_MFCC, HOP_LENGTH, N_FFT, N_MELS,
)
from src.data_loader import load_dataset, get_class_distribution
from src.feature_extractor import FeatureExtractor
from src.feature_extraction import extract_mfcc, save_features
from src.augmentation import augment_waveforms_in_memory
from src.gmm_classifier import GMMClassifier
from src.svm_classifier import SVMClassifier
from src.hmm_model import HMMClassifier
from src.rf_classifier import RFClassifier
from src.xgb_classifier import XGBCryClassifier
from src.ensemble_classifier import EnsembleClassifier
from src.evaluation import full_evaluation, plot_model_comparison, plot_tsne, plot_feature_importance

for d in [FEATURES_DIR, MODELS_DIR, RESULTS_DIR, PLOTS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('All imports successful.')
print(f'Classes: {CLASSES}')
print(f'Sample rate: {SAMPLE_RATE} Hz, Duration: {DURATION}s')
print(f'Random state: {RANDOM_STATE}, Test size: {TEST_SIZE}')

---
## Step 3: Load ONLY Original Audio (No Augmented Files)
This is the critical data leakage fix — we exclude all `_aug_` files so the train/test split is clean.

In [ ]:
print('Loading ORIGINAL audio files only (excluding _aug_ files)...')
print('=' * 60)

X_raw, y, file_paths = load_dataset(originals_only=True)

print(f'\nTotal original files loaded: {len(y)}')
print(f'Class distribution: {get_class_distribution(y)}')

aug_leaked = [f for f in file_paths if '_aug_' in f]
assert len(aug_leaked) == 0, f'DATA LEAK: {len(aug_leaked)} augmented files found!'
print(f'Augmented files in dataset: {len(aug_leaked)} (verified clean)')

---
## Step 4: Split Originals into Train/Test BEFORE Augmentation
The test set will contain ONLY original, never-augmented recordings.

In [ ]:
print(f'Splitting originals: {TEST_SIZE:.0%} test, {1-TEST_SIZE:.0%} train')
print('=' * 60)

indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    indices, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)

X_train_raw = X_raw[train_idx]
X_test_raw = X_raw[test_idx]
y_train_orig = y[train_idx]
y_test = y[test_idx]

train_paths = [file_paths[i] for i in train_idx]
test_paths = [file_paths[i] for i in test_idx]

print(f'Train originals: {len(y_train_orig)}')
print(f'Test  originals: {len(y_test)}')
print(f'\nTrain distribution: {get_class_distribution(y_train_orig)}')
print(f'Test  distribution: {get_class_distribution(y_test)}')

test_aug = [p for p in test_paths if '_aug_' in p]
print(f'\nAugmented files in TEST set: {len(test_aug)} (must be 0)')

# Save split info for Phase 2 reuse
import pandas as pd
split_df = pd.DataFrame({
    'filepath': [file_paths[i] for i in train_idx] + [file_paths[i] for i in test_idx],
    'label': [CLASSES[y[i]] for i in train_idx] + [CLASSES[y[i]] for i in test_idx],
    'split': ['train'] * len(train_idx) + ['test'] * len(test_idx),
})
split_path = RESULTS_DIR / 'split_indices.csv'
split_df.to_csv(split_path, index=False)
print(f'\nSplit saved to {split_path} (reusable in Phase 2)')

---
## Step 5: Augment ONLY the Training Set (In Memory)
Minority classes in the training set are augmented to ~300 samples each. The test set is untouched.

In [ ]:
AUG_TARGET = 300

print(f'Augmenting training set in memory (target: {AUG_TARGET}/class)...')
print('=' * 60)

aug_waveforms, aug_labels = augment_waveforms_in_memory(
    waveforms=list(X_train_raw),
    labels=y_train_orig,
    target_per_class=AUG_TARGET,
)

X_train_all = np.concatenate(
    [X_train_raw, np.array(aug_waveforms, dtype=np.float32)], axis=0
)
y_train = np.concatenate([y_train_orig, aug_labels], axis=0)

print(f'\nTraining set composition:')
print(f'  Originals:  {len(y_train_orig)}')
print(f'  Augmented:  {len(aug_labels)}')
print(f'  Total:      {len(y_train)}')
print(f'  Distribution: {get_class_distribution(y_train)}')
print(f'\nTest set (untouched originals): {len(y_test)}')
print(f'  Distribution: {get_class_distribution(y_test)}')

del aug_waveforms, aug_labels
gc.collect()

---
## Step 6: Extract 411-Dimensional Features
Features: 240 MFCC + 120 CQCC + 7 Pitch/F0 + 14 Spectral Contrast + 24 Chroma + 6 Spectral Stats

In [ ]:
extractor = FeatureExtractor()
feature_names = None

def extract_features_for_set(waveforms, desc='Extracting'):
    global feature_names
    clip_list = []
    mfcc_seqs = []
    for waveform in tqdm(waveforms, desc=desc):
        try:
            feats, names = extractor.extract_all(waveform)
            clip_list.append(feats)
            if feature_names is None:
                feature_names = names
        except Exception as exc:
            print(f'[ERROR] {exc}')
            clip_list.append(np.zeros(411, dtype=np.float32))
        mfcc_mat = extract_mfcc(waveform)
        mfcc_seqs.append(mfcc_mat.T)
    return np.array(clip_list, dtype=np.float32), mfcc_seqs

print('Extracting TRAINING features...')
X_train, train_mfcc_seqs = extract_features_for_set(X_train_all, desc='Train features')
print(f'Train feature shape: {X_train.shape}')
del X_train_all, X_train_raw
gc.collect()

print('\nExtracting TEST features...')
X_test, test_mfcc_seqs = extract_features_for_set(X_test_raw, desc='Test features')
print(f'Test feature shape: {X_test.shape}')
del X_test_raw, X_raw
gc.collect()

# Save features to Drive (reusable in Phase 2)
save_features(X_train, y_train, tag='train')
save_features(X_test, y_test, tag='test')
np.save(FEATURES_DIR / 'y_test.npy', y_test)
np.save(FEATURES_DIR / 'y_train.npy', y_train)
print(f'\nFeatures saved to {FEATURES_DIR}/ (persisted on Drive)')

---
## Step 7: Train & Evaluate All Models
Training 6 classifiers: GMM, SVM, HMM, Random Forest, XGBoost, Ensemble (Stacking)

Each model is in its own cell so you can see results immediately.

In [ ]:
all_metrics = {}
print('=' * 60)
print('  MODEL TRAINING & EVALUATION')
print('=' * 60)

In [ ]:
# --- GMM ---
print('--- GMM Classifier (Bayesian, 8 components, diag covariance) ---')
gmm = GMMClassifier(n_components=8, covariance_type='diag')
gmm.fit(X_train, y_train)
gmm.save()
gmm_preds = gmm.predict(X_test)
gmm_proba = gmm.predict_proba(X_test)
all_metrics['GMM'] = full_evaluation(y_test, gmm_preds, 'GMM', y_proba=gmm_proba)
print(f"  Accuracy: {all_metrics['GMM']['accuracy']:.4f}  |  Macro F1: {all_metrics['GMM']['macro_f1']:.4f}")
gc.collect()

In [ ]:
# --- SVM ---
print('--- SVM Classifier (RBF + SMOTE + Grid Search) ---')
svm = SVMClassifier()
svm.fit_with_grid_search(X_train, y_train, cv=5, scoring='f1_macro')
svm.save()
svm_preds = svm.predict(X_test)
svm_proba = svm.predict_proba(X_test)
all_metrics['SVM'] = full_evaluation(y_test, svm_preds, 'SVM', y_proba=svm_proba)
print(f"  Accuracy: {all_metrics['SVM']['accuracy']:.4f}  |  Macro F1: {all_metrics['SVM']['macro_f1']:.4f}")
gc.collect()

In [ ]:
# --- HMM ---
print('--- HMM Classifier (left-right, 8 states) ---')
hmm = HMMClassifier(n_states=8, covariance_type='diag')
hmm.fit(train_mfcc_seqs, y_train)
hmm.save()
hmm_preds = hmm.predict(test_mfcc_seqs)
all_metrics['HMM'] = full_evaluation(y_test, hmm_preds, 'HMM')
print(f"  Accuracy: {all_metrics['HMM']['accuracy']:.4f}  |  Macro F1: {all_metrics['HMM']['macro_f1']:.4f}")
gc.collect()

In [ ]:
# --- Random Forest ---
print('--- Random Forest Classifier (500 trees) ---')
rf = RFClassifier(n_estimators=500)
rf.fit(X_train, y_train)
rf.save()
rf_preds = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)
all_metrics['RF'] = full_evaluation(y_test, rf_preds, 'RF', y_proba=rf_proba)
print(f"  Accuracy: {all_metrics['RF']['accuracy']:.4f}  |  Macro F1: {all_metrics['RF']['macro_f1']:.4f}")
gc.collect()

In [ ]:
# --- XGBoost ---
print('--- XGBoost Classifier (300 rounds) ---')
xgb = XGBCryClassifier(n_estimators=300)
xgb.fit(X_train, y_train)
xgb.save()
xgb_preds = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)
all_metrics['XGBoost'] = full_evaluation(y_test, xgb_preds, 'XGBoost', y_proba=xgb_proba)
print(f"  Accuracy: {all_metrics['XGBoost']['accuracy']:.4f}  |  Macro F1: {all_metrics['XGBoost']['macro_f1']:.4f}")
gc.collect()

In [ ]:
# --- Ensemble (Stacking) ---
print('--- Ensemble Classifier (Stacking: SVM + RF + XGBoost) ---')
base_models = {'SVM': svm, 'RF': rf, 'XGBoost': xgb}
ensemble = EnsembleClassifier()
ensemble.fit(X_train, y_train, base_models=base_models)
ensemble.save()
ens_preds = ensemble.predict(X_test)
ens_proba = ensemble.predict_proba(X_test)
all_metrics['Ensemble'] = full_evaluation(y_test, ens_preds, 'Ensemble', y_proba=ens_proba)
print(f"  Accuracy: {all_metrics['Ensemble']['accuracy']:.4f}  |  Macro F1: {all_metrics['Ensemble']['macro_f1']:.4f}")
gc.collect()

---
## Step 8: Results Summary

In [ ]:
print('\n' + '=' * 70)
print('  FINAL RESULTS (LEAK-FREE EVALUATION)')
print('=' * 70)
print(f'  Test set: {len(y_test)} ORIGINAL samples (no augmented data)')
print(f'  Test distribution: {get_class_distribution(y_test)}')
print()
print(f"  {'Model':<12s} {'Acc':>7s} {'MacF1':>7s} {'WgtF1':>7s} "
      f"{'MCC':>7s} {'Kappa':>7s} {'AUC':>7s}")
print('  ' + '-' * 58)

best_model = None
best_f1 = 0

for name, m in all_metrics.items():
    auc = f"{m['auc_roc']:.4f}" if m.get('auc_roc') is not None else '  N/A'
    print(f"  {name:<12s} {m['accuracy']:>7.4f} {m['macro_f1']:>7.4f} "
          f"{m['weighted_f1']:>7.4f} {m['mcc']:>7.4f} {m['kappa']:>7.4f} "
          f"{auc:>7s}")
    if m['macro_f1'] > best_f1:
        best_f1 = m['macro_f1']
        best_model = name

print(f'\n  Best model: {best_model} (Macro F1 = {best_f1:.4f})')

# Save summary to Drive
summary_path = RESULTS_DIR / 'final_summary.json'
with open(summary_path, 'w') as f:
    json.dump(all_metrics, f, indent=2, default=str)
print(f'  Summary saved -> {summary_path}')

---
## Step 9: Per-Class Breakdown

In [ ]:
print(f'Per-class breakdown for {best_model}:')
print('=' * 60)
print(f"  {'Class':<15s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Support':>10s}")
print('  ' + '-' * 55)

best_m = all_metrics[best_model]
for cls_name, cm in best_m['per_class'].items():
    print(f"  {cls_name:<15s} {cm['precision']:>10.4f} "
          f"{cm['recall']:>10.4f} {cm['f1']:>10.4f} "
          f"{int(cm['support']):>10d}")

---
## Step 10: Visualizations

In [ ]:
plot_model_comparison(all_metrics)
plt.show()

all_features = np.concatenate([X_train, X_test], axis=0)
all_labels = np.concatenate([y_train, y_test], axis=0)
plot_tsne(all_features, all_labels)
plt.show()

rf_imp = rf.feature_importances(feature_names)
plot_feature_importance(rf_imp, model_name='RF', top_n=20)
plt.show()

xgb_imp = xgb.feature_importances(feature_names)
plot_feature_importance(xgb_imp, model_name='XGBoost', top_n=20)
plt.show()

print('All plots saved to results/plots/ on Drive')

---
## Step 11: Top Features

In [ ]:
print('Top 15 features (Random Forest):')
for i, (fname, imp) in enumerate(rf_imp[:15], 1):
    print(f'  {i:2d}. {fname:<35s}  {imp:.4f}')

print('\nTop 15 features (XGBoost):')
for i, (fname, imp) in enumerate(xgb_imp[:15], 1):
    print(f'  {i:2d}. {fname:<35s}  {imp:.4f}')

---
## Step 12: Verify Drive Persistence
Everything is already saved on Google Drive. For Phase 2, you only need to:
1. Mount Drive (same Step 0 cell)
2. Load the saved features and split from `features/` and `results/split_indices.csv`
3. Add your Phase 2 notebook/code to the same Drive folder

In [ ]:
print('Files saved on Google Drive (persistent across sessions):')
print('=' * 60)
for folder in ['data/raw', 'src', 'features', 'models', 'results/metrics', 'results/plots']:
    p = Path(PROJECT_DIR) / folder
    if p.exists():
        count = len(list(p.iterdir()))
        print(f'  {folder + "/":30s} {count} files')

print(f'\nFor Phase 2, mount Drive and set:')
print(f"  PROJECT_DIR = '{DRIVE_PROJECT}'")
print(f'  Then load features with np.load(FEATURES_DIR / ...)')
print(f'  And reuse the same train/test split from results/split_indices.csv')